# Libraries and Utility Functions

In this problem, you will implement a a Kalman filter for state estimation of all joints of the 7DOF Kuka iiwa robot. We will assume that a range-and-bearing sensor is located at the end effector of the robot for our measurement model and that there are several beacons in the workspace.

In [ ]:
# Run once at the beginning to install pybullet
!pip install pybullet

In [ ]:
import pybullet as p
import pybullet_data
import numpy as np
from scipy.stats import norm
from unittest.mock import patch

import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML  # Required for rendering animation in Colab

# Configure Matplotlib default for JSHTML
plt.rcParams["animation.html"] = "jshtml"
plt.rcParams['figure.dpi'] = 90
plt.ioff()

In [ ]:
# Utility functions for kinematics calculations

def get_ee_pos(q_vec):
    orig_states = [p.getJointState(robot_id, i)[0] for i in range(7)]

    # Set candidate joint states
    p.resetJointState(robot_id, active_joints[0], q_vec[0])
    p.resetJointState(robot_id, active_joints[1], q_vec[1])

    # Force PyBullet to compute FK for candidate state
    ee_pos = np.array(p.getLinkState(robot_id, ee_link_idx, computeForwardKinematics=1)[0])

    # Restore original joint states and refresh cache
    for idx, pos in enumerate(orig_states):
        p.resetJointState(robot_id, idx, pos)
    p.getLinkState(robot_id, ee_link_idx, computeForwardKinematics=1)

    return ee_pos


def get_ee_jacobian(q, robot_id, ee_link_idx):
    """
    Computes and returns the 2x7 linear position Jacobian for all 7 active joints.
    """
    zero_vec = [0.0] * 7

    lin_jac, _ = p.calculateJacobian(
        bodyUniqueId=robot_id,
        linkIndex=ee_link_idx,
        localPosition=[0, 0, 0],
        objPositions=list(q),
        objVelocities=zero_vec,
        objAccelerations=zero_vec
    )

    return np.array(lin_jac)[:2]

In [ ]:
# Static visualization function

def visualize_7dof_errors(history, dt=0.05):
    time = np.arange(len(history)) * dt

    true_q = np.array([h['true_q'] for h in history])   # Shape (N, 7)
    est_q = np.array([h['est_q'] for h in history])     # Shape (N, 7)
    true_ee = np.array([h['true_ee'] for h in history]) # Shape (N, 2 or 3)
    est_ee = np.array([h['est_ee'] for h in history])   # Shape (N, 2 or 3)

    # 1. Compute per-joint RMSE & joint errors
    q_errors = true_q - est_q  # Shape (N, 7)
    q_rmse = np.sqrt(np.mean(q_errors**2, axis=0))  # Vector of 7 RMSE values

    ee_errors = true_ee - est_ee
    ee_rmse = np.sqrt(np.mean(ee_errors**2, axis=0))

    fig, (ax_err, ax_ee) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

    # --- Top Subplot: Joint Angle Tracking Errors ---
    colors = plt.cm.tab10(np.linspace(0, 1, 7))
    for i in range(7):
        ax_err.plot(
            time, q_errors[:, i],
            color=colors[i], lw=1.5,
            label=f'Joint {i} (RMSE: {q_rmse[i]:.3f} rad)'
        )

    ax_err.axhline(0, color='black', linestyle=':', alpha=0.7)
    ax_err.set_ylabel("Joint Error $q_{true} - q_{est}$ (rad)", fontsize=10)
    ax_err.set_title("7-DOF Joint Space Estimation Errors", fontsize=12, fontweight='bold')
    ax_err.grid(True, linestyle='--', alpha=0.5)
    # 2-column legend prevents overcrowding
    ax_err.legend(loc='upper right', fontsize=8, ncol=2, framealpha=0.9)

    # --- Bottom Subplot: Cartesian Position Tracking ---
    ax_ee.plot(time, true_ee[:, 0], 'g-', lw=2, label='X True')
    ax_ee.plot(time, est_ee[:, 0], 'm--', lw=2, label=f'X Est (RMSE: {ee_rmse[0]:.3f} m)')
    ax_ee.plot(time, true_ee[:, 1], 'b-', lw=2, label='Y True')
    ax_ee.plot(time, est_ee[:, 1], 'c--', lw=2, label=f'Y Est (RMSE: {ee_rmse[1]:.3f} m)')

    ax_ee.set_xlabel("Time (seconds)", fontsize=10)
    ax_ee.set_ylabel("End-Effector Position (m)", fontsize=10)
    ax_ee.set_title("Cartesian End-Effector Tracking", fontsize=11, fontweight='bold')
    ax_ee.grid(True, linestyle='--', alpha=0.5)
    ax_ee.legend(loc='upper right', fontsize=8, ncol=2, framealpha=0.9)

    plt.tight_layout()
    plt.show()

In [ ]:
# Animation functions

from matplotlib.patches import Ellipse

def plot_covariance_ellipse(mean, cov, ax, n_std=2.0, edgecolor='magenta', facecolor='magenta', alpha=0.2, label=None):
    """
    Plots a 2D Gaussian confidence ellipse (default 2-sigma ~ 95% confidence).
    """
    # Eigen-decomposition of 2x2 covariance matrix
    vals, vecs = np.linalg.eigh(cov[:2, :2])

    # Sort eigenvalues/vectors descending
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]

    # Clamp small negative eigenvalues caused by numerical precision
    vals = np.maximum(vals, 0.0)

    # Compute ellipse geometry
    theta = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    width = 2 * n_std * np.sqrt(vals[0])
    height = 2 * n_std * np.sqrt(vals[1])

    ellipse = Ellipse(
        xy=mean[:2], width=width, height=height, angle=theta,
        edgecolor=edgecolor, facecolor=facecolor, alpha=alpha, lw=1.5, label=label
    )
    ax.add_patch(ellipse)


def animate_ekf(frame):
    ax_joint.cla()
    ax_work.cla()

    # --- Subplot 1: Joint Space (q0 vs q1) ---
    ax_joint.set_xlim(-np.pi, np.pi)
    ax_joint.set_ylim(-np.pi, np.pi)
    ax_joint.grid(True, linestyle='--', alpha=0.5)
    ax_joint.set_title(f"Joint Space (q0 vs q1) | Step {frame}", fontsize=11)
    ax_joint.set_xlabel("Joint 0 Angle (rad)")
    ax_joint.set_ylabel("Joint 1 Angle (rad)")

    # Extract current state and covariance estimate
    est_q_curr = history[frame]['est_q']
    P_q_curr = history[frame]['P_est']

    # Draw 2-Sigma Joint Space Covariance Ellipse
    plot_covariance_ellipse(
        mean=est_q_curr[:2], cov=P_q_curr[:2, :2], ax=ax_joint,
        n_std=2.0, edgecolor='magenta', facecolor='magenta', alpha=0.25,
        label='$2\\sigma$ Covariance ($P_q$)'
    )

    true_q_traj = np.array([history[i]['true_q'] for i in range(frame + 1)])
    est_q_traj = np.array([history[i]['est_q'] for i in range(frame + 1)])

    ax_joint.plot(true_q_traj[:, 0], true_q_traj[:, 1], 'g-', lw=2, label='True q')
    ax_joint.plot(est_q_traj[:, 0], est_q_traj[:, 1], 'm--', lw=2, label='Estimated q')
    ax_joint.plot(true_q_traj[-1, 0], true_q_traj[-1, 1], 'go', ms=7)
    ax_joint.plot(est_q_traj[-1, 0], est_q_traj[-1, 1], 'mo', ms=7)
    ax_joint.legend(loc='upper right', fontsize=8)

    # --- Subplot 2: Workspace (X vs Y) ---
    ax_work.set_xlim(-1.2, 1.2)
    ax_work.set_ylim(-1.2, 1.2)
    ax_work.set_aspect('equal')
    ax_work.grid(True, linestyle='--', alpha=0.5)
    ax_work.set_title(f"Workspace (X vs Y) | Step {frame}", fontsize=11)
    ax_work.set_xlabel("X Position (m)")
    ax_work.set_ylabel("Y Position (m)")

    ax_work.scatter(beacons[:, 0], beacons[:, 1], c='red', marker='^', s=100, label='Beacons', zorder=5)

    est_ee_curr = history[frame]['est_ee']
    P_ee_curr = history[frame]['P_ee']

    plot_covariance_ellipse(
        mean=est_ee_curr[:2], cov=P_ee_curr, ax=ax_work,
        n_std=2.0, edgecolor='magenta', facecolor='magenta', alpha=0.25,
        label='$2\\sigma$ Mapped Covariance ($P_{ee}$)'
    )

    true_ee_traj = np.array([history[i]['true_ee'] for i in range(frame + 1)])
    est_ee_traj = np.array([history[i]['est_ee'] for i in range(frame + 1)])

    ax_work.plot(true_ee_traj[:, 0], true_ee_traj[:, 1], 'g-', lw=2, label='True EE')
    ax_work.plot(est_ee_traj[:, 0], est_ee_traj[:, 1], 'm--', lw=2, label='Estimated EE')
    ax_work.plot(true_ee_traj[-1, 0], true_ee_traj[-1, 1], 'go', ms=7)
    ax_work.plot(est_ee_traj[-1, 0], est_ee_traj[-1, 1], 'mo', ms=7)
    ax_work.legend(loc='upper right', fontsize=8)

    plt.tight_layout()

# Extended Kalman Filter

It is difficult to scale the particle filter up to higher dimensions, which is why we restricted its usage to just the first two joints. If we are willing to restrict the belief state to a multivariate Gaussian distribution, then the (extended) Kalman filter can be applied to a very high-dimensional state space.

The setup of the Kalman filter class below follows that of the particle filter. `x_est` is a 1D array storing the estimated mean joint configuration, and `P_est` is a 2D array storing the corresponding covariance matrix. In each step, the subroutines are `predict_ekf()` and `update_ekf()`, both of which you will implement.

In [ ]:
class ExtendedKalmanFilter7DOF:
    def __init__(self, initial_q=np.zeros(7), initial_P=np.eye(7) * 0.1):
        self.x_est = np.array(initial_q, dtype=float)
        self.P_est = np.array(initial_P, dtype=float)

    def step(self, control_u, z_measurements, dt, motion_noise, measure_noise_r, measure_noise_b):
        self.x_est, self.P_est = predict_ekf(
            self.x_est, self.P_est, control_u, dt, motion_noise
        )
        self.x_est, self.P_est = update_ekf(
            self.x_est, self.P_est, z_measurements, measure_noise_r, measure_noise_b
        )

    def estimate(self):
        return self.x_est.copy()

In [ ]:
def simulate_kalman_filter_7dof(motion_noise, measure_noise_r, measure_noise_b, num_steps=200, dt=0.05):
    ekf = ExtendedKalmanFilter7DOF()
    history = []
    substeps = int(dt / (1.0 / 240.0))
    noise_r_std = 0.04
    noise_b_std = 0.05

    for step in range(num_steps):
        if step == 100:
            bump_joint_idx = active_joints[0]
            current_angle = p.getJointState(robot_id, bump_joint_idx)[0]
            p.resetJointState(robot_id, bump_joint_idx, current_angle - 0.5)

        t = step * dt
        # Generate target velocities for all 7 joints
        target_velocities = np.array([
            0.8 * np.sin(1.2 * t),
            0.6 * np.cos(0.8 * t),
            0.4 * np.sin(1.0 * t),
            -0.4 * np.cos(1.1 * t),
            0.3 * np.sin(0.9 * t),
            -0.3 * np.cos(0.7 * t),
            0.2 * np.sin(0.5 * t)
        ])
        noisy_target_velocities = target_velocities + np.random.normal(0, 0.03, size=len(target_velocities))

        for idx, j_idx in enumerate(active_joints):
            p.setJointMotorControl2(
                robot_id, j_idx, p.VELOCITY_CONTROL,
                targetVelocity=noisy_target_velocities[idx], force=200
            )

        for _ in range(substeps):
            p.stepSimulation()

        q_true = np.array([p.getJointState(robot_id, j)[0] for j in active_joints])
        true_ee_pos = np.array(p.getLinkState(robot_id, ee_link_idx, computeForwardKinematics=1)[0])

        dx = beacons[:, 0] - true_ee_pos[0]
        dy = beacons[:, 1] - true_ee_pos[1]

        true_ranges = np.sqrt(dx**2 + dy**2)
        true_bearings = np.arctan2(dy, dx)

        z_ranges_noisy = true_ranges + np.random.normal(0, noise_r_std, size=len(beacons))
        raw_noisy_bearings = true_bearings + np.random.normal(0, noise_b_std, size=len(beacons))
        z_bearings_noisy = np.arctan2(np.sin(raw_noisy_bearings), np.cos(raw_noisy_bearings))
        z_noisy = np.column_stack([z_ranges_noisy, z_bearings_noisy])

        ekf.step(target_velocities, z_noisy, dt, motion_noise, measure_noise_r, measure_noise_b)
        q_est = ekf.estimate()
        est_ee_pos = get_ee_pos(q_est)

        # Pre-compute workspace covariance while connected to PyBullet
        J_pos = get_ee_jacobian(q_est, robot_id, ee_link_idx)[:2, :2]
        P_ee = J_pos @ ekf.P_est[:2, :2] @ J_pos.T

        history.append({
            'true_q': q_true.copy(),
            'est_q': q_est.copy(),
            'P_est': ekf.P_est.copy(),
            'P_ee': P_ee.copy(),
            'true_ee': true_ee_pos[:2],
            'est_ee': est_ee_pos[:2]
        })

    return history

# 4.1: Motion Prediction (3 pts)

We will use the same velocity motion model as in the particle filter, which is given by $\mathbf x \leftarrow \mathbf x + \mathbf u dt$. (There are no joint limits here, as we cannot accommodate any modification to the Gaussian assumptions of the Kalman filter.)

The Jacobian of the motion model is the identity, so the covariance matrix is just updated by adding to it the noise covariance $Q$. You can simply take this to be a diagonal matrix containing identical $\sigma^2$ entries.

In [ ]:
def predict_ekf(x_est, P_est, u_control, dt, std_noise):
    """
    Predict step for 7-DOF arm under velocity control.
    Returns: Predicted mean and covariance matrix
    """
    # TODO
    return x_est, P_est

# 4.2: Beacon Jacobian (6 pts)

`compute_beacon_jacobian()` will compute the Jacobian of the measurement model relating the range and bearing w.r.t. joint positions. Given a beacon position `beacon_pos`, compute $\mathbf H$ in the same way as derived in class. This is a $2 \times 2$ matrix, since we have no $\theta$ column. This relates the measurements to the end effector.

To map the measurements to the joint angles, simply compose $\mathbf H$ with the usual arm Jacobian $\mathbf J$ retrieved from `get_ee_jacobian()`: $\mathbf H_q = \mathbf H \mathbf J$. The final result should be a $2 \times N$ matrix.

In [ ]:
def compute_beacon_jacobian(x_est, beacon_pos):
    """Computes and returns the 2xN measurement Jacobian Matrix H_q for a
    single beacon using the chain rule: H_q = (dh/dp_ee) @ J_pos_2d
    """
    # TODO
    pass

# 4.3: Innovation (6 pts)

Next, we have a procedure computing the innovation. Given a range and bearing measurement in `z_beacon`, we need the difference between this and the *expected* range and bearing. This is almost identical to that of the particle filter, with one crucial difference: The $\Delta\phi$ component must be correctly *signed*. We cannot rely on symmetry since we are not drawing density values from a PDF. Thus, $\Delta\phi$ should be defined as follows:

$$\Delta\phi = \begin{cases} \phi - \hat\phi, & \text{if } |\phi - \hat\phi| \leq \pi \\
(\phi - \hat\phi) + 2\pi, & \text{if } \phi - \hat\phi < -\pi \\
(\phi - \hat\phi) - 2\pi, & \text{if } \phi - \hat\phi > \pi \end{cases}$$

The innovation can then be returned as the vector (array) $\mathbf y = (\Delta r, \Delta \phi)$.

In [ ]:
def compute_innovation(x_est, z_beacon, beacon_pos):
    """Computes the 2D measurement innovation vector y = z - h(x) for a single beacon,
    applying signed shortest angle wrapping to the bearing residual.
    """
    # TODO
    pass

# 4.3: Measurement Update (6 pts)

With procedures deriving the Jacobian and innovation, the remainder of the measurement update should be straightforward. Simply iterate over each beacon, compute the necessary quantities, and apply the appropriate updates to `x_est` and `P_est`. Return both quantities when done.

In [ ]:
def update_ekf(x_est, P_est, z_measurements, measure_noise_r, measure_noise_b):
    """Performs sequential EKF measurement updates across all beacons by delegating
    Jacobian and innovation calculations to dedicated functions.
    """
    # TODO
    return x_est, P_est

As with the particle filter, here is a unit test for the full measurement update.

In [ ]:
# --- Helper mocks for offline testing ---
def mock_get_ee_pos(x):
    return np.array([x[0], x[1], 0.0])

def mock_get_ee_jacobian(x, robot_id, ee_link_idx):
    # 2x2 position Jacobian (top 2 rows are 2x2 identity)
    return np.array([[1.0, 0.0], [0.0, 1.0]])

TEST_MOCKS = {
    "beacons": np.array([[1.0, 0.0], [-1.0, 0.01]]),
    "get_ee_pos": mock_get_ee_pos,
    "get_ee_jacobian": mock_get_ee_jacobian,
    "robot_id": 0,  # Dummy PyBullet ID
    "ee_link_idx": 6,  # Dummy link index
    "active_joints": [0, 1],  # Dummy active joint list
}


# --- 1. Test Jacobian Computation ---
def test_compute_beacon_jacobian():
    x_test = np.array([0.0, 0.0])
    beacon_0 = np.array([1.0, 0.0])

    # For beacon at [1, 0] and EE at [0, 0]: dx=-1, dy=0, r=1
    # dh_dp = [[-1, 0], [0, -1]] -> H = dh_dp @ I_2x2
    expected_H = np.array([[-1.0, 0.0], [0.0, -1.0]])

    with patch.dict(globals(), TEST_MOCKS):
        H = compute_beacon_jacobian(x_test, beacon_0)

    assert H.shape == (2, 2), f"Expected shape (2, 2), got {H.shape}"
    assert np.allclose(H, expected_H), (
        f"Jacobian mismatch!\nGot:\n{H}\nExpected:\n{expected_H}"
    )
    print("✓ test_compute_beacon_jacobian passed!")


# --- 2. Test Innovation & Angle Wrapping ---
def test_compute_innovation():
    x_test = np.array([0.0, 0.0])
    beacon_1 = np.array([-1.0, 0.01])  # expected bearing ~ +3.13159 rad

    # Measurement bearing -3.15159 rad tests -pi boundary wrapping
    z_beacon = np.array([1.00005, -3.15159])

    # Exact expected innovation y = z - h(x)
    expected_y = np.array([1.24993771e-09, 2.32027646e-06])

    with patch.dict(globals(), TEST_MOCKS):
        y = compute_innovation(x_test, z_beacon, beacon_1)

    assert np.allclose(y, expected_y, atol=1e-8), (
        f"Innovation mismatch!\nGot: {y}\nExpected: {expected_y}"
    )
    print("✓ test_compute_innovation passed!")


# --- 3. Test Full Sequential Update ---
def test_update_ekf():
    x_pred = np.array([0.0, 0.0])
    P_pred = np.eye(2) * 0.1
    z_measurements = np.array([[1.0, 0.0], [1.00005, -3.15159]])

    expected_x_est = np.array([9.8018e-09, 1.0939e-06])
    expected_P_est = np.array(
        [[3.8924e-03, 9.0638e-06], [9.0638e-06, 5.7050e-03]]
    )

    with patch.dict(globals(), TEST_MOCKS):
        x_est, P_est = update_ekf(
            x_pred,
            P_pred,
            z_measurements,
            measure_noise_r=0.09,
            measure_noise_b=0.11,
        )

    assert np.allclose(x_est, expected_x_est, atol=1e-5), (
        f"State estimate mismatch!\nGot: {x_est}\nExpected: {expected_x_est}"
    )
    assert np.allclose(P_est, expected_P_est, atol=1e-5), (
        f"Covariance estimate mismatch!\nGot:\n{P_est}\nExpected:\n{expected_P_est}"
    )
    print("✓ test_update_ekf passed!")


# Run full suite
test_compute_beacon_jacobian()
test_compute_innovation()
test_update_ekf()

# 4.4: Experimentation (4 pts)

We can now test the Kalman filter. First run the cell below with the code provided. You should see that the filter tracks the robot reasonably well, with plots showing both the C-space and workspace progression. There is a disturbance to one of the joints at 5 seconds, but the filter is able to somewhat recover.

Experiment with separately setting low and high values of the motion noise and measure noise parameters. For example, set motion noise to $0$, and then to $0.3$. Then set it back to $0.03$, and set measurement noise (both range and bearing) to $0.01$, and then to $1.0$.

For each setting, briefly compare your observations with that of the baseline result. Also give a reason **explaining** why the filter performs differently for each. For submission, run and leave the output of the setting with the original parameter values of $0.03, 0.04, 0.05$.

[Describe your observations here.]

In [ ]:
# Run the 7-DOF simulation
np.random.seed(100)

sim_id = p.connect(p.DIRECT)
p.setAdditionalSearchPath(pybullet_data.getDataPath())

robot_id = p.loadURDF("kuka_iiwa/model.urdf", useFixedBase=True, physicsClientId=sim_id)
p.setGravity(0, 0, -9.81, physicsClientId=sim_id)
p.setRealTimeSimulation(1, physicsClientId=sim_id)

active_joints = list(range(7))  # Enable all 7 joints
ee_link_idx = 6

# Fixed workspace beacons (x, y, z)
beacons = np.array([
    [0.6,  0.6, 0.5],
    [-0.6, 0.6, 0.5],
    [0.0, -0.6, 0.5]
])

motion_noise = 0.03
measure_noise_r = 0.04
measure_noise_b = 0.05
history = simulate_kalman_filter_7dof(motion_noise, measure_noise_r, measure_noise_b)
p.disconnect()

# Display errors for all 7 joints
visualize_7dof_errors(history)

You can also view an animation of the Kalman filter in both the C-space of the first two joints and the workspace. The Gaussian is represented by the 2-std ellipse. Seeing how it evolves over time should help confirm your hypotheses about Kalman filter behavior under different scenarios.

In [ ]:
fig, (ax_joint, ax_work) = plt.subplots(1, 2, figsize=(13, 6))
anim = animation.FuncAnimation(fig, animate_ekf, frames=len(history), interval=60)
plt.close(fig)

HTML(anim.to_jshtml())